# Shortest distance from the origin to 100 pizza slices
### One problem, three formulations, one lesson about *how you write it down*

A **pizza slice** (circular sector) $S_i$ is the intersection of a disk and two half-planes through the disk's centre (the tip):

$$S_i=\Big\{x\in\mathbb R^2:\ \|x-c_i\|\le R_i\ \ \text{and}\ \ \big|\angle(x-c_i)-\beta_i\big|\le\delta_i\Big\}.$$

The tip is $c_i$, the radius is $R_i$, the wedge points along $\beta_i$ with half-width $\delta_i$. As long as the wedge is narrower than a half-plane ($2\delta_i<\pi$) the two angular bounds are ordinary linear inequalities, so **each slice is a convex set** (disk $\cap$ two half-planes).

We want the distance from the origin to the **union** of 100 such slices:

$$d^\star=\min_{1\le i\le 100}\ \min_{x\in S_i}\ \|x\|.$$

We solve it three ways and **animate the optimiser's progress** for each:

| | formulation | convexity | solver |
|---|---|---|---|
|**1**| 100 independent per-slice problems, take the min | each is **convex** | **Clarabel** (conic interior point, via CVXPY) |
|**2**| one program over $x\in\mathbb R^2$, constrained to the **union** | **nonconvex** (disconnected feasible set) | **Ipopt** (nonlinear interior point, via CasADi) |
|**3**| the same program in **polar** coordinates $x=\rho(\cos\alpha,\sin\alpha)$ | nonconvex, and — the hypothesis — *"even worse"* | **Ipopt** |

All three chase the *same* number $d^\star$. The questions are which formulations actually **find** it (§5) and what each one **costs** (§4).

### Setup
This notebook uses two state-of-the-art solvers, both pip-installable:

```bash
pip install numpy scipy matplotlib cvxpy clarabel casadi
```

`cvxpy`+`clarabel` handle the convex cone programs; `casadi` ships **Ipopt** (with exact automatic derivatives and a per-iteration callback we use to record the optimiser's path for the animations).

In [1]:
import numpy as np
import cvxpy as cp
import casadi as ca
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams["figure.figsize"] = (5.7, 5.7)

## 0. The scene

100 random sectors with tips scattered in an annulus around the origin. We keep every tip farther from the origin than its own radius ($\|c_i\|>R_i$), which guarantees the **origin lies strictly outside every slice**, so all distances are positive and the problem is non-trivial. Wedges are drawn with half-width $\delta_i\in[15^\circ,70^\circ]$, i.e. $2\delta_i<\pi$, so **each slice is convex**.

In [2]:
def make_slices(n=100, seed=7):
    rng   = np.random.default_rng(seed)
    tip_r = rng.uniform(3.0, 12.0, n)                       # tip distance from origin
    tip_a = rng.uniform(0, 2*np.pi, n)
    c     = np.stack([tip_r*np.cos(tip_a), tip_r*np.sin(tip_a)], axis=1)
    R     = rng.uniform(0.4, 1.5, n)                        # R < tip_r  =>  origin outside disk
    beta  = rng.uniform(0, 2*np.pi, n)                      # wedge bisector
    delta = rng.uniform(np.deg2rad(15), np.deg2rad(70), n)  # half-width, 2*delta < pi -> convex
    return dict(c=c, R=R, beta=beta, delta=delta, n=n)

def slice_normals(beta, delta):
    '''Inward normals of a wedge's two bounding half-planes.'''
    lo, hi = beta - delta, beta + delta
    n_lo = np.array([-np.sin(lo),  np.cos(lo)])
    n_hi = np.array([ np.sin(hi), -np.cos(hi)])
    return n_lo, n_hi

S = make_slices(100, seed=7)
print(f"{S['n']} slices; origin outside all: {np.all(np.linalg.norm(S['c'],axis=1) > S['R'])}")

100 slices; origin outside all: True


### A closed-form ground truth

For a *convex* sector the nearest point to the origin is elementary: it is the closest of the two bounding radii (segments) and — if the origin's direction falls inside the wedge — the arc. We compute it in closed form to have an **independent check** on the numerical solvers, and a `which_slice` helper that tells us *which* slice a converged point landed on (used later to score the nonconvex methods without being fooled by smoothing bias).

In [3]:
def _proj_segment(p, a, b):
    ab = b - a
    t  = np.clip(np.dot(p - a, ab) / np.dot(ab, ab), 0.0, 1.0)
    return a + t * ab

def dist_point_to_sector(p, c, R, beta, delta):
    d_lo = np.array([np.cos(beta - delta), np.sin(beta - delta)])
    d_hi = np.array([np.cos(beta + delta), np.sin(beta + delta)])
    v, nrm = p - c, np.linalg.norm(p - c)
    if 1e-12 < nrm <= R:                       # possibly inside
        if abs(((np.arctan2(v[1],v[0]) - beta + np.pi) % (2*np.pi)) - np.pi) <= delta:
            return 0.0
    cands = [_proj_segment(p, c, c + R*d_lo), _proj_segment(p, c, c + R*d_hi)]
    if nrm > 1e-12:                            # arc candidate if direction is inside the wedge
        if abs(((np.arctan2(v[1],v[0]) - beta + np.pi) % (2*np.pi)) - np.pi) <= delta:
            cands.append(c + R * v/nrm)
    return float(min(np.linalg.norm(z - p) for z in cands))

def which_slice(p, S):
    ds = [dist_point_to_sector(p, S["c"][i], S["R"][i], S["beta"][i], S["delta"][i])
          for i in range(S["n"])]
    return int(np.argmin(ds))

def ground_truth(S):
    dists = np.array([dist_point_to_sector(np.zeros(2), S["c"][i], S["R"][i],
                                           S["beta"][i], S["delta"][i]) for i in range(S["n"])])
    i = int(np.argmin(dists))
    return dict(dists=dists, i_star=i, d_star=float(dists[i]), x_star=_nearest_point(S, i))

def _nearest_point(S, i):
    c,R,beta,delta = S["c"][i],S["R"][i],S["beta"][i],S["delta"][i]
    d_lo = np.array([np.cos(beta-delta),np.sin(beta-delta)])
    d_hi = np.array([np.cos(beta+delta),np.sin(beta+delta)])
    cands=[_proj_segment(np.zeros(2),c,c+R*d_lo),_proj_segment(np.zeros(2),c,c+R*d_hi)]
    v=-c;
    if abs(((np.arctan2(v[1],v[0])-beta+np.pi)%(2*np.pi))-np.pi)<=delta:
        cands.append(c+R*v/np.linalg.norm(v))
    return min(cands,key=lambda z:np.dot(z,z))

gt = ground_truth(S)
print(f"closed-form ground truth:  d* = {gt['d_star']:.6f}   at slice {gt['i_star']}   x* = {gt['x_star']}")

closed-form ground truth:  d* = 1.752146   at slice 32   x* = [-0.93947913  1.47898372]


## 1. Convex decomposition — 100 problems, solved in parallel

For a single slice the problem is convex,

$$\min_x\ \|x\|^2 \quad\text{s.t.}\quad \|x-c_i\|\le R_i,\qquad (x-c_i)^\top n^{lo}_i\ge 0,\qquad (x-c_i)^\top n^{hi}_i\ge 0,$$

a small **second-order cone program** that Clarabel solves to the *global* optimum every time. There is no coupling between slices, so the 100 problems are embarrassingly parallel; $d^\star$ is just the minimum of the 100 answers. This is our reliable reference.

In [4]:
def solve_convex_clarabel(S):
    dists = np.empty(S["n"]); pts = np.empty((S["n"], 2))
    for i in range(S["n"]):
        c, R = S["c"][i], S["R"][i]
        n_lo, n_hi = slice_normals(S["beta"][i], S["delta"][i])
        x = cp.Variable(2)
        cons = [cp.norm(x - c) <= R, (x - c) @ n_lo >= 0, (x - c) @ n_hi >= 0]
        cp.Problem(cp.Minimize(cp.sum_squares(x)), cons).solve(solver=cp.CLARABEL)
        pts[i], dists[i] = x.value, np.linalg.norm(x.value)
    i = int(np.argmin(dists))
    return dict(dists=dists, pts=pts, i_star=i, d_star=float(dists[i]), x_star=pts[i])

clara = solve_convex_clarabel(S)
err = np.abs(clara["dists"] - gt["dists"]).max()
print(f"Clarabel:  d* = {clara['d_star']:.6f}  at slice {clara['i_star']}")
print(f"max |Clarabel - closed form| over all 100 slices = {err:.2e}   ->  solvers agree")

Clarabel:  d* = 1.752146  at slice 32
max |Clarabel - closed form| over all 100 slices = 2.84e-08   ->  solvers agree


To animate the *progress* of this method on the same footing as the nonconvex ones, we re-solve each convex subproblem with **Ipopt** and record its iterates through a CasADi callback. Because each subproblem is convex, Ipopt lands on exactly the point Clarabel certified — but now we have a geometric trajectory to watch.

In [5]:
class IterRecorder(ca.Callback):
    '''Logs the primal iterate x at every Ipopt iteration.'''
    def __init__(self, name, nx, ng):
        ca.Callback.__init__(self); self.nx, self.ng, self.log = nx, ng, []
        self.construct(name, {})
    def get_n_in(self):  return ca.nlpsol_n_out()
    def get_n_out(self): return 1
    def get_name_in(self, i):  return ca.nlpsol_out(i)
    def get_name_out(self, i): return "ret"
    def get_sparsity_in(self, i):
        nm = ca.nlpsol_out(i)
        if nm == "f": return ca.Sparsity.scalar()
        if nm in ("x", "lam_x"): return ca.Sparsity.dense(self.nx)
        if nm in ("g", "lam_g"): return ca.Sparsity.dense(self.ng)
        return ca.Sparsity(0, 0)
    def eval(self, arg):
        self.log.append(np.array(ca.DM(arg[0])).flatten().copy()); return [0]

def convex_ipopt_paths(S):
    paths, pts = [], np.empty((S["n"], 2))
    for i in range(S["n"]):
        c, R = S["c"][i], S["R"][i]
        n_lo, n_hi = slice_normals(S["beta"][i], S["delta"][i])
        x  = ca.MX.sym("x", 2); xc = x - c
        g  = ca.vertcat(ca.dot(xc, xc) - R**2, -ca.dot(xc, n_lo), -ca.dot(xc, n_hi))
        rec = IterRecorder(f"r{i}", 2, 3)
        sol = ca.nlpsol("s", "ipopt", {"x": x, "f": ca.dot(x, x), "g": g},
                        {"iteration_callback": rec, "ipopt.print_level": 0,
                         "print_time": 0, "ipopt.tol": 1e-8, "ipopt.sb": "yes"})
        x0 = c + 0.4*R*np.array([np.cos(S["beta"][i]), np.sin(S["beta"][i])])   # interior start
        r  = sol(x0=x0, lbg=-ca.inf, ubg=0.0)
        pts[i] = np.array(r["x"]).flatten(); paths.append(np.array(rec.log))
    return paths, pts

paths1, pts1 = convex_ipopt_paths(S)
print(f"recorded {len(paths1)} trajectories; iterations/slice: "
      f"min {min(map(len,paths1))}, max {max(map(len,paths1))}")

recorded 100 trajectories; iterations/slice: min 7, max 13


#### Shared plotting helpers

In [6]:
TMOVE, THOLD = 18, 6
T = TMOVE + THOLD                      # frames per animation
FIGSIZE, DPI = (5.7, 5.7), 74
C_SLICE,C_EDGE,C_WIN,C_WINE = "#e9e6df","#c9c2b4","#ffcf33","#e0a800"
C_ORIGIN,C_DOT,C_HIT,C_MISS,C_LINE = "#111","#3b6ea5","#2a9d4a","#c1443c","#d1495b"

def resample(path, T):
    path = np.asarray(path, float)
    if path.ndim != 2 or len(path) == 0: return np.zeros((T, 2))
    idx = np.clip(np.round(np.linspace(0, 1, T)*(len(path)-1)).astype(int), 0, len(path)-1)
    return path[idx]

def draw_scene(ax, S, gt):
    lim = max(np.abs(S["c"]).max() + 2, 4)
    for i in range(S["n"]):
        c, R = S["c"][i], S["R"][i]
        b, d = np.rad2deg(S["beta"][i]), np.rad2deg(S["delta"][i]); win = (i == gt["i_star"])
        ax.add_patch(Wedge(c, R, b-d, b+d, facecolor=C_WIN if win else C_SLICE,
                           edgecolor=C_WINE if win else C_EDGE, lw=1.4 if win else 0.6,
                           zorder=3 if win else 1, alpha=1 if win else 0.9))
    ax.plot(0, 0, 'o', color=C_ORIGIN, ms=7, zorder=6)
    ax.plot(0, 0, 'o', mfc='none', mec=C_ORIGIN, ms=13, zorder=6)
    ax.annotate("origin", (0, 0), xytext=(6, -14), textcoords="offset points", fontsize=8)
    ax.plot(*gt["x_star"], '*', color=C_WINE, ms=16, mec='k', mew=.6, zorder=7)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_edgecolor("#bbb")

### Animation 1 — 100 convex solves converge together
Each dot is one slice's optimiser walking to that slice's nearest point; the red line tracks the current best. Every dot descends **monotonically** and no dot is ever trapped — that is what convexity buys you. The gold star is the true global optimum.

In [7]:
def anim_method1(S, gt, paths):
    P = np.array([resample(p, T) for p in paths])                 # (100, T, 2)
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI); draw_scene(ax, S, gt)
    scat = ax.scatter(P[:,0,0], P[:,0,1], s=16, c=C_DOT, alpha=.55, zorder=5, edgecolors='none')
    best, = ax.plot([], [], '-', color=C_LINE, lw=2, zorder=6)
    ttl = ax.set_title("", fontsize=11)
    def upd(t):
        Q = P[:, t, :]; scat.set_offsets(Q)
        dd = np.linalg.norm(Q, axis=1); j = int(np.argmin(dd))
        best.set_data([0, Q[j,0]], [0, Q[j,1]])
        ttl.set_text(f"Method 1 · 100 convex subproblems in parallel\n"
                     f"all converge · best d = {dd[j]:.3f}   (global d* = {gt['d_star']:.3f})")
        return scat, best, ttl
    return FuncAnimation(fig, upd, frames=T, interval=150), fig

a1, f1 = anim_method1(S, gt, paths1); plt.close(f1)
HTML(a1.to_jshtml(fps=6))

## 2. One nonconvex program in Cartesian coordinates

Now we insist on a **single** optimisation variable $x\in\mathbb R^2$ that must lie *somewhere* in the union. Membership in slice $i$ means all three of its constraint functions are $\le 0$; membership in the union means that holds for **at least one** slice:

$$x\in\bigcup_i S_i\iff \min_i\ \underbrace{\max_k\ g_{i,k}(x)}_{\text{inside slice }i\ \Leftrightarrow\ \le 0}\ \le\ 0,\qquad
g_{i,\cdot}(x)=\big(\|x-c_i\|-R_i,\ -( x-c_i)^\top n^{lo}_i,\ -(x-c_i)^\top n^{hi}_i\big).$$

The `min` over slices is what makes the feasible set **nonconvex** — it is a union of disjoint convex pieces. We give Ipopt a smooth version using a **soft-max / soft-min** (log-sum-exp) with small temperatures $\tau,\mu$; a local solver then converges to the boundary of *whichever* slice its basin leads to. From a single start it can easily lock onto a nearby-but-wrong slice, so we run **many random starts**.

In [8]:
def _soft_max(v, tau):  M = ca.mmax(v); return M + tau*ca.log(ca.sum1(ca.exp((v - M)/tau)))
def _soft_min(v, mu):   m = ca.mmin(v); return m - mu*ca.log(ca.sum1(ca.exp(-(v - m)/mu)))

def union_constraint(x, S, tau=0.05, mu=0.05):
    '''Smooth G(x): <= 0 means x is inside the union of the slices.'''
    per_slice = []
    for i in range(S["n"]):
        c = S["c"][i]; R = S["R"][i]
        n_lo, n_hi = slice_normals(S["beta"][i], S["delta"][i]); xc = x - c
        g_i = ca.vertcat(ca.sqrt(ca.dot(xc, xc) + 1e-9) - R, -ca.dot(xc, n_lo), -ca.dot(xc, n_hi))
        per_slice.append(_soft_max(g_i, tau))
    return _soft_min(ca.vertcat(*per_slice), mu)

def build_cartesian(S, tau=0.05, mu=0.05):
    x = ca.MX.sym("x", 2); rec = IterRecorder("c", 2, 1)
    solver = ca.nlpsol("s", "ipopt", {"x": x, "f": ca.dot(x, x), "g": union_constraint(x,S,tau,mu)},
                       {"iteration_callback": rec, "ipopt.print_level": 0, "print_time": 0,
                        "ipopt.tol": 1e-6, "ipopt.max_iter": 300, "ipopt.sb": "yes"})
    return solver, rec

def run_cartesian(solver, rec, x0):
    rec.log.clear(); r = solver(x0=x0, lbg=-ca.inf, ubg=0.0)
    xs = np.array(r["x"]).flatten()
    return dict(x=xs, dist=float(np.linalg.norm(xs)), path=np.array(rec.log))

### Animation 2 — 24 random starts, Cartesian
Green trajectories reach the **global** slice; red ones get **trapped** on some other slice. Watch how many end up red.

In [9]:
K = 24
X0 = np.random.default_rng(5).uniform(-11, 11, (K, 2))     # shared starts (reused by Method 3)

def anim_multistart(S, gt, runs, num, coord):
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI); draw_scene(ax, S, gt)
    paths = [resample(r["path"], T) for r in runs]
    lines, heads = [], []
    for r in runs:
        col = C_HIT if r["hit"] else C_MISS
        ln, = ax.plot([], [], '-', color=col, lw=1.3, alpha=.7, zorder=4)
        hd, = ax.plot([], [], 'o', color=col, ms=5, zorder=5)
        lines.append(ln); heads.append(hd)
    nhit = sum(r["hit"] for r in runs); ttl = ax.set_title("", fontsize=11)
    def upd(t):
        for ln, hd, Pth in zip(lines, heads, paths):
            ln.set_data(Pth[:t+1,0], Pth[:t+1,1]); hd.set_data([Pth[t,0]], [Pth[t,1]])
        ttl.set_text(f"Method {num} · one nonconvex program ({coord}) · {len(runs)} random starts\n"
                     f"reached global slice: {nhit}/{len(runs)}   "
                     f"(green = global · red = trapped)")
        return (*lines, *heads, ttl)
    return FuncAnimation(fig, upd, frames=T, interval=150), fig, nhit

solver_c, rec_c = build_cartesian(S)
cart = [run_cartesian(solver_c, rec_c, x0) for x0 in X0]
for r in cart: r["hit"] = (which_slice(r["x"], S) == gt["i_star"])
a2, f2, n2 = anim_multistart(S, gt, cart, 2, "Cartesian"); plt.close(f2)
print(f"Cartesian: {n2}/{K} starts reached the global slice")
HTML(a2.to_jshtml(fps=6))

Cartesian: 6/24 starts reached the global slice


## 3. The same program in polar coordinates

Write $x=\rho(\cos\alpha,\sin\alpha)$ and optimise over $(\rho,\alpha)$. The objective collapses to $\rho^2$, but the *same* union constraint now sees curved, periodic geometry. The original intuition: this should be **even less convex** — each slice's region becomes nonconvex in $(\rho,\alpha)$, $\alpha$ is $2\pi$-periodic (many separated angular bands), and there is a coordinate singularity at $\rho=0$. Let's test it with the **identical starting points** used in Method 2.

In [10]:
def build_polar(S, tau=0.05, mu=0.05, rho_max=15.0):
    z = ca.MX.sym("z", 2); rho, alpha = z[0], z[1]
    x = ca.vertcat(rho*ca.cos(alpha), rho*ca.sin(alpha)); rec = IterRecorder("p", 2, 1)
    solver = ca.nlpsol("s", "ipopt", {"x": z, "f": rho**2, "g": union_constraint(x,S,tau,mu)},
                       {"iteration_callback": rec, "ipopt.print_level": 0, "print_time": 0,
                        "ipopt.tol": 1e-6, "ipopt.max_iter": 300, "ipopt.sb": "yes"})
    return solver, rec, rho_max

def run_polar(solver, rec, rho0, a0, rho_max):
    rec.log.clear()
    r = solver(x0=[rho0, a0], lbx=[0, -ca.inf], ubx=[rho_max, ca.inf], lbg=-ca.inf, ubg=0.0)
    z = np.array(r["x"]).flatten(); xy = np.array([z[0]*np.cos(z[1]), z[0]*np.sin(z[1])])
    log = np.array(rec.log)
    xpath = (np.stack([log[:,0]*np.cos(log[:,1]), log[:,0]*np.sin(log[:,1])], 1)
             if len(log) else log)
    return dict(x=xy, dist=float(z[0]), path=xpath)

solver_p, rec_p, rmax = build_polar(S)
pol = [run_polar(solver_p, rec_p, np.linalg.norm(x0), np.arctan2(x0[1], x0[0]), rmax) for x0 in X0]
for r in pol: r["hit"] = (which_slice(r["x"], S) == gt["i_star"])
a3, f3, n3 = anim_multistart(S, gt, pol, 3, "polar (rho, alpha)"); plt.close(f3)
print(f"Cartesian reached global: {n2}/{K}   |   Polar reached global: {n3}/{K}")
HTML(a3.to_jshtml(fps=6))

Cartesian reached global: 6/24   |   Polar reached global: 10/24


## 4. What does each formulation cost?

We time the **solve** itself (the one-time CasADi build of the 100-term union constraint is reported separately). Two things tend to surprise people:

- A **single** nonconvex Ipopt solve is the *cheapest* operation of all — a few milliseconds — because it touches the geometry once, from one starting point. Method 1 does about **100× more** solver work, since it must examine every slice.
- But raw speed is the wrong yardstick. That single cheap solve reaches the global optimum only ~25–45% of the time **and never tells you whether it did**. You buy reliability with restarts, and the bill climbs — yet even 24 starts come with *no certificate*. Method 1's 100 convex solves are each solved to global optimality **with a duality certificate**, and they are independent, so on $p$ cores the wall-clock falls by ~$p$.

So the honest ranking is not *"Method 1 is cheapest"* but **"Method 1 is the cheapest route to an answer you can trust"** — and that gap only widens as slices grow more numerous or higher-dimensional, where nonconvex basins multiply while each convex piece stays easy.

In [11]:
import time
def timeit(fn, rep=3):
    fn()                                                   # warm-up
    t0 = time.perf_counter()
    for _ in range(rep): fn()
    return (time.perf_counter() - t0) / rep

t1  = timeit(lambda: solve_convex_clarabel(S), rep=3)                       # 100 convex solves
tb2 = time.perf_counter(); build_cartesian(S); tb2 = time.perf_counter()-tb2   # one-time build
tb3 = time.perf_counter(); build_polar(S);     tb3 = time.perf_counter()-tb3
x0 = np.array([5.0, 5.0])
t2  = timeit(lambda: run_cartesian(solver_c, rec_c, x0), rep=10)            # one nonconvex solve
t3  = timeit(lambda: run_polar(solver_p, rec_p, np.linalg.norm(x0),
                               np.arctan2(x0[1], x0[0]), rmax), rep=10)
tm2 = timeit(lambda: [run_cartesian(solver_c, rec_c, x) for x in X0], rep=2)   # 24-start search
tm3 = timeit(lambda: [run_polar(solver_p, rec_p, np.linalg.norm(x),
                                np.arctan2(x[1], x[0]), rmax) for x in X0], rep=2)

print(f"{'':32s}{'time':>10s}   reliability")
print(f"{'-'*70}")
print(f"{'Method 1  100 convex solves':32s}{t1*1e3:8.1f} ms   GLOBAL + certificate "
      f"({t1/100*1e3:.2f} ms/slice, parallelizable)")
print(f"{'Method 2  one Ipopt solve':32s}{t2*1e3:8.1f} ms   ~25% global from one start, no certificate")
print(f"{'Method 2  24-start search':32s}{tm2*1e3:8.1f} ms   {sum(r['hit'] for r in cart)}/24 global, still no certificate")
print(f"{'Method 3  one Ipopt solve':32s}{t3*1e3:8.1f} ms   ~45% global from one start, no certificate")
print(f"{'Method 3  24-start search':32s}{tm3*1e3:8.1f} ms   {sum(r['hit'] for r in pol)}/24 global, still no certificate")
print(f"\none-time CasADi solver build: Method 2 {tb2*1e3:.0f} ms, Method 3 {tb3*1e3:.0f} ms")
print("(a single nonconvex solve is cheapest; Method 1 is the cheapest *trustworthy* answer.)")

                                      time   reliability
----------------------------------------------------------------------
Method 1  100 convex solves        206.6 ms   GLOBAL + certificate (2.07 ms/slice, parallelizable)
Method 2  one Ipopt solve            2.8 ms   ~25% global from one start, no certificate
Method 2  24-start search          169.7 ms   6/24 global, still no certificate
Method 3  one Ipopt solve            4.2 ms   ~45% global from one start, no certificate
Method 3  24-start search          163.5 ms   10/24 global, still no certificate

one-time CasADi solver build: Method 2 51 ms, Method 3 50 ms
(a single nonconvex solve is cheapest; Method 1 is the cheapest *trustworthy* answer.)


## 5. What actually happened — and why it's the real lesson

Look at the two nonconvex animations side by side. In **Cartesian** the trajectories thrash between slices and most stall on the wrong one. In **polar** the paths are visibly **radial**: $\rho$ slides straight in toward the origin while $\alpha$ rotates to hunt for the nearest ray — and **more of them reach the global slice.** The coordinate change that was supposed to make things *worse* made the solver *better*.

Repeating the multi-start over 200 shared random starts on five different scenes (scoring by *which slice* each run landed on, so smoothing bias can't distort it):

| scene (seed) | $d^\star$ | **Cartesian** global-hit | **Polar** global-hit | mean $(d-d^\star)$: Cart → Polar |
|---|---|---|---|---|
| 7  | 1.75 | 20% | **47%** | 2.28 → 0.67 |
| 11 | 1.93 | 25% | **48%** | 1.95 → 0.89 |
| 23 | 1.72 | 15% | **34%** | 2.86 → 1.14 |
| 42 | 1.82 |  6% | **21%** | 2.42 → 1.23 |
| 99 | 2.06 | 35% | **43%** | 2.19 → 0.91 |

Polar roughly **doubles** the global-hit rate and **halves** the average suboptimality — consistently.

**Why?** It is the *objective*, not the constraint geometry. In Cartesian, $\|x\|^2$ makes every lateral move between slices cost objective value, so the disconnected feasible pieces become isolated local minima with small basins — the solver sticks. In polar the objective is $\rho^2$, **independent of $\alpha$**: rotating between angular sectors is *free* as far as the objective is concerned. That decouples *which direction* ($\alpha$, objective-flat) from *how far* ($\rho$, the thing being minimised), giving the solver cheap lateral mobility to escape toward the globally nearest ray. The genuine downsides you'd predict — nonconvex slice regions, $2\pi$ periodicity, the $\rho=0$ singularity — are real, which is why polar *still* fails 50–80% of the time; they just don't outweigh the free rotation.

### Takeaways
- **Only Method 1 is guaranteed.** Decomposing a hard problem into convex pieces and combining the answers beats any single nonconvex formulation for reliability. When you *can* decompose, do.
- **"More/less convex" is the wrong mental model** for a nonconvex problem. What a local solver actually sees is the *number and size of basins*, which depends jointly on the objective, the constraints, the conditioning, **and the coordinates** — and it is *testable*, not guessable. Here the intuition pointed the wrong way.
- A practical pattern this suggests: use a cheap smooth/global-ish search (Method 2 or 3) to pick the promising slice, then **polish** with one convex solve (Method 1) on that slice to get the exact answer.